# Task 4: First Machine Learning Classifier / Predictor
**Tools:** Python, Scikit-Learn  
**Dataset:** California Housing (built into Scikit-Learn)  
**Goal:** Build and evaluate Linear Regression and Decision Tree models to predict house prices, and a Decision Tree classifier to predict whether a house is high-priced.

## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

sns.set_theme(style='whitegrid')

data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.shape)
df.head()

Target `MedHouseVal` is the median house value in units of **$100,000**.

In [ ]:
df.describe().T

In [ ]:
print(df.isnull().sum().sum(), 'missing values')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df['MedHouseVal'], bins=40, color='#2E86AB', ax=axes[0])
axes[0].set_title('House Value Distribution')
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1], annot_kws={'size': 7})
axes[1].set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Train/test split

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 3. Model training (regression): predict house prices

In [ ]:
models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeRegressor(max_depth=8, random_state=42))]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print('Trained:', name)

## 4. Evaluation metrics (MAE, MSE, RMSE, R²)

In [ ]:
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2 (test)': r2_score(y_test, pred),
        'R2 (train)': r2_score(y_train, model.predict(X_train)),
    })
results = pd.DataFrame(rows).set_index('Model').round(3)
results

A large gap between train and test R² means overfitting. `max_depth=8` keeps the tree from memorising the training data.

In [ ]:
best_name = results['R2 (test)'].idxmax()
best_model = models[best_name]
pred = best_model.predict(X_test)
print('Best model:', best_name)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, pred, alpha=0.3, color='#2E86AB')
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, color='#E4572E', linewidth=2)
axes[0].set_xlabel('Actual ($100k)')
axes[0].set_ylabel('Predicted ($100k)')
axes[0].set_title(f'Actual vs Predicted ({best_name})')

sns.histplot(y_test - pred, bins=40, color='#6A4C93', ax=axes[1])
axes[1].set_title('Residuals (Actual - Predicted)')
plt.tight_layout()
plt.show()

In [ ]:
tree = models['Decision Tree'].named_steps['model']
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
imp.plot(kind='barh', color='#2E86AB', ax=ax)
ax.set_title('Decision Tree Feature Importance')
plt.tight_layout()
plt.show()

## 5. Classification: is the house high-priced?
Turn the target into two classes: `1` = value above the median, `0` = value at or below the median. This lets us use **Accuracy** and a **Confusion Matrix**.

In [ ]:
y_cls = (y > y.median()).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

clf = Pipeline([('scaler', StandardScaler()),
                ('model', DecisionTreeClassifier(max_depth=6, random_state=42))])
clf.fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)

print(f"Accuracy (test) : {accuracy_score(yc_test, yc_pred):.3f}")
print(f"Accuracy (train): {accuracy_score(yc_train, clf.predict(Xc_train)):.3f}")
print()
print(classification_report(yc_test, yc_pred, target_names=['Lower price', 'Higher price']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    yc_test, yc_pred, display_labels=['Lower', 'Higher'], cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()
print(confusion_matrix(yc_test, yc_pred))

## 6. Save the trained model
The saved file is loaded by the Streamlit app in Task 5.

In [ ]:
joblib.dump(best_model, 'model.joblib')
print('Saved model.joblib using', best_name)

# Quick test: reload and predict
loaded = joblib.load('model.joblib')
sample = X_test.iloc[[0]]
print('Predicted price: $', round(float(loaded.predict(sample)[0]) * 100000))
print('Actual price   : $', round(float(y_test.iloc[0]) * 100000))

## Conclusion
- Trained and compared Linear Regression and Decision Tree models with a train/test split.
- Evaluated with MAE, MSE, RMSE and R² for regression, and Accuracy, Confusion Matrix and a classification report for classification.
- The Decision Tree usually beats Linear Regression because it captures non-linear patterns (such as location effects). Median income is the most important feature.
- The best model was saved with joblib for deployment.

*(Check these statements against your own outputs.)*